# Supplementary Figure — Low-TMB TMB axis score vs slope and correlation

Two scatter panels per cancer type:
1. Median TMB axis score in low-TMB tumors vs TMB axis slope (from `tableS2/axis_score_slope.csv`)
2. Median TMB axis score in low-TMB tumors vs TMB axis correlation (from `score_corr_per_type_summary.csv`)

Output: `results/figures/supplementary/supp_fig2/tmb_axis_baseline_vs_slope_and_corr.pdf`

In [ ]:
import os
import socket
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

hostname = socket.gethostname()
if "scg" in hostname:
    BASE_DIR = "/labs/mwinslow/Karen/protein-master/v2_pancan_batch_corrected"
else:
    BASE_DIR = os.path.expanduser(
        "~/Documents/research/scg/protein-master/v2_pancan_batch_corrected"
    )

DATA_DIR = f"{BASE_DIR}/analysis/01_axis_analysis/bh/axis_sample_scores/tcga_xena"
TABLE_DIR = f"{BASE_DIR}/results/tables/tableS2"
CORR_PATH = f"{BASE_DIR}/results/tables/score_corr_per_type_summary.csv"
SLOPE_PATH = f"{TABLE_DIR}/axis_score_slope.csv"
OUT_DIR = f"{BASE_DIR}/results/figures/supplementary/supp_fig2"
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, f"{BASE_DIR}/local/figures")
from palette import COLORS, rasterize_figure

In [ ]:
# Low-TMB baseline: median TMB axis score in lower within-type TMB bin
pos_axis = pd.read_parquet(f"{DATA_DIR}/axis_sample_level.parquet")
pos_axis["tmb_bin"] = pos_axis.groupby("cancer_type")["log_mut"].transform(
    lambda x: pd.qcut(x, q=2, labels=["low", "high"], duplicates="drop")
)
low_tmb_axis = (
    pos_axis.loc[pos_axis["tmb_bin"] == "low"]
    .groupby("cancer_type")["TMB_axis_score"]
    .median()
    .rename("low_tmb_axis_score")
)

# Precomputed per-cancer slope and correlation
tmb_slope = (
    pd.read_csv(SLOPE_PATH)
    .loc[lambda d: d["score"] == "TMB_axis_score",
         ["cancer_type", "slope", "slope_ci_low", "slope_ci_high", "n"]]
)

tmb_corr = (
    pd.read_csv(CORR_PATH)
    .loc[lambda d: d["score"] == "TMB_axis_score",
         ["cancer_type", "r", "ci_low", "ci_high"]]
    .rename(columns={"r": "tmb_corr", "ci_low": "tmb_corr_ci_low", "ci_high": "tmb_corr_ci_high"})
)

df_plot = (
    tmb_slope
    .set_index("cancer_type")
    .join(low_tmb_axis)
    .join(tmb_corr.set_index("cancer_type"))
    .reset_index()
    .dropna(subset=["low_tmb_axis_score", "slope", "tmb_corr"])
)

df_plot.head()

In [ ]:
from matplotlib.gridspec import GridSpec

CI_COLOR = COLORS["light_gray"]


def _format_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def _baseline_scatter(ax, df, y_col, ci_low_col, ci_high_col, ylabel, title):
    r_pearson, p_pearson = pearsonr(df["low_tmb_axis_score"], df[y_col])
    r_spearman, p_spearman = spearmanr(df["low_tmb_axis_score"], df[y_col])

    ax.errorbar(
        df["low_tmb_axis_score"],
        df[y_col],
        yerr=[
            df[y_col] - df[ci_low_col],
            df[ci_high_col] - df[y_col],
        ],
        fmt="o",
        color=COLORS["low"],
        ecolor=CI_COLOR,
        elinewidth=0.8,
        capsize=1.5,
        markersize=4.5,
        markeredgecolor="black",
        markeredgewidth=0.4,
    )

    sns.regplot(
        data=df,
        x="low_tmb_axis_score",
        y=y_col,
        scatter=False,
        ci=None,
        color=COLORS["dark_gray"],
        line_kws={"linewidth": 0.8, "alpha": 0.7},
        ax=ax,
    )

    for _, row in df.iterrows():
        ax.text(
            row["low_tmb_axis_score"],
            row[y_col],
            row["cancer_type"],
            fontsize=6,
            ha="left",
            va="bottom",
        )

    ax.axhline(0, color=COLORS["light_gray"], linewidth=0.7, linestyle="--")
    ax.axvline(
        df["low_tmb_axis_score"].median(),
        color=COLORS["light_gray"],
        linewidth=0.7,
        linestyle="--",
    )

    ax.annotate(
        f"Pearson r = {r_pearson:.2f}\n{_format_p(p_pearson)}\n"
        f"Spearman rho = {r_spearman:.2f}\n{_format_p(p_spearman)}",
        xy=(0.02, 0.98),
        xycoords="axes fraction",
        va="top",
        ha="left",
        fontsize=8,
    )

    ax.set_xlabel("Low-TMB TMB axis score")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=10)
    sns.despine(ax=ax)


def plot_baseline_vs_slope_and_corr(df, filename):
    panel_size = 5.0
    fig = plt.figure(figsize=(panel_size * 2, panel_size))
    gs = GridSpec(1, 2, figure=fig, wspace=0.28)

    ax_slope = fig.add_subplot(gs[0, 0])
    _baseline_scatter(
        ax_slope,
        df,
        y_col="slope",
        ci_low_col="slope_ci_low",
        ci_high_col="slope_ci_high",
        ylabel="Slope (TMB response)",
        title="Baseline TMB axis vs TMB response",
    )

    ax_corr = fig.add_subplot(gs[0, 1])
    _baseline_scatter(
        ax_corr,
        df,
        y_col="tmb_corr",
        ci_low_col="tmb_corr_ci_low",
        ci_high_col="tmb_corr_ci_high",
        ylabel="Pearson r (TMB axis vs log TMB)",
        title="Baseline TMB axis vs TMB correlation",
    )

    fig.subplots_adjust(left=0.07, right=0.99, top=0.95, bottom=0.1)
    rasterize_figure(fig)
    outpath = f"{OUT_DIR}/{filename}.pdf"
    fig.savefig(outpath, dpi=300, bbox_inches="tight", pad_inches=0.08)
    plt.show()
    print(f"Saved: {outpath}")


plot_baseline_vs_slope_and_corr(df_plot, "tmb_axis_baseline_vs_slope_and_corr")